# High-rise 01 - Inlet profile validation

Validate the atmospheric boundary layer: detect the vertical profiles in the
probe cloud, plot mean velocity / turbulence intensity / spectrum into `debug/`,
and extract the simulation mean velocity at the reference height (`u_ref`). That
`u_ref` feeds the Cp non-dimensionalisation in notebook 02.

Defaults run on the `pitot_inlet` fixture. Point at a real case by setting the
`CFDMOD_HR_INFLOW_HIST` / `CFDMOD_HR_INFLOW_POINTS` / `CFDMOD_HR_REF_HEIGHT`
environment variables (or editing the config cell).

In [ ]:
import os
import pathlib

import numpy as np  # noqa: F401  (used across later cells)

import matplotlib

matplotlib.use("Agg")  # headless: notebooks write files, they do not display

from cfdmod import inflow_report, mesh_field, plot_config  # noqa: E402
from cfdmod.building import (  # noqa: E402
    BuildingCase,
    cf_per_floor,
    cm_per_floor,
    cp_from_pressure,
    example_building_case,
    example_building_structure,
    floor_accelerations,
    floor_load_source,
    peak_response_table,
    peak_value,
    plot_floor_mass,
    plot_mode_shape,
    plot_natural_frequencies,
    solve_building_response,
    structure_from_csvs,
)
from cfdmod.report import DebugWriter  # noqa: E402


def _find_repo(start: pathlib.Path) -> pathlib.Path:
    p = start.resolve()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    return start.resolve()


REPO = _find_repo(pathlib.Path.cwd())

plot_config.apply_style()

FIX = REPO / "fixtures" / "tests"
OUTPUT_BASE = pathlib.Path(
    os.environ.get("CFDMOD_HR_OUTPUT_BASE", REPO / "examples" / "high_rise" / "_run")
)
VERSION = os.environ.get("CFDMOD_HR_VERSION", "example")
print("REPO:", REPO)
print("OUTPUT_BASE:", OUTPUT_BASE, "| VERSION:", VERSION)

In [ ]:
from cfdmod.inflow import InflowData, NormalizationParameters

# --- config -------------------------------------------------------------
HIST = pathlib.Path(
    os.environ.get(
        "CFDMOD_HR_INFLOW_HIST", FIX / "inflow" / "pitot_inlet" / "hist_series.csv"
    )
)
POINTS = pathlib.Path(
    os.environ.get("CFDMOD_HR_INFLOW_POINTS", FIX / "inflow" / "pitot_inlet" / "points.csv")
)
REFERENCE_HEIGHT = float(os.environ.get("CFDMOD_HR_REF_HEIGHT", "2.0"))
COMPONENT = os.environ.get("CFDMOD_HR_COMPONENT", "ux")

dbg = DebugWriter(OUTPUT_BASE, stage="inflow", version=VERSION)
inflow = InflowData.from_files(HIST, POINTS)
profiles = inflow_report.detect_profiles(inflow, min_points=3)
print(f"detected {len(profiles)} vertical profile(s)")
for p in profiles:
    print(f"  {p.name}: {len(p.point_idx)} points, z in [{p.z.min():.2f}, {p.z.max():.2f}]")

In [ ]:
# --- per-profile figures + reference velocity ---------------------------
u_ref_by_profile = {}
for prof in profiles:
    u_ref = inflow_report.reference_velocity(prof, inflow, REFERENCE_HEIGHT, component=COMPONENT)
    u_ref_by_profile[prof.name] = u_ref
    L = inflow_report.integral_length_scale(
        inflow, prof.nearest_index(REFERENCE_HEIGHT), u_ref, component=COMPONENT
    )
    print(f"{prof.name}: u_ref(z={REFERENCE_HEIGHT:g}) = {u_ref:.4f} m/s | L = {L:.4g} m")

    norm = NormalizationParameters(
        reference_velocity=max(u_ref, 1e-9), characteristic_length=1.0
    )
    for name, fig in {
        "mean_velocity": inflow_report.plot_mean_velocity(prof, inflow, component=COMPONENT),
        "turbulence_intensity": inflow_report.plot_turbulence_intensity(
            prof, inflow, component=COMPONENT
        ),
        "spectrum": inflow_report.plot_spectrum(
            prof, inflow, REFERENCE_HEIGHT, norm, component=COMPONENT
        ),
    }.items():
        dbg.savefig(fig, f"{prof.name}/{name}.png")
        plot_config.close(fig)
print("figures written under", dbg.debug_dir)

In [ ]:
# --- code-standard comparison (NBR 6123 / EN 1991-1-4) ------------------
# Overlay the simulated mean velocity / turbulence intensity on the code
# curves, and compare the integral length scale to the EN 1991-1-4 theory.
CAT_EU = os.environ.get("CFDMOD_HR_CAT_EU", "III")
Z0 = float(os.environ.get("CFDMOD_HR_Z0", "0.3"))
prof = profiles[0]
fig, _ = inflow_report.plot_profile_vs_code(
    prof, inflow, REFERENCE_HEIGHT, cat_eu=CAT_EU, component=COMPONENT
)
dbg.savefig(fig, f"{prof.name}/profile_vs_code.png", deliverable=True)
plot_config.close(fig)

L_num = inflow_report.integral_length_scale_profile(inflow, prof, component=COMPONENT)
L_eu = inflow_report.eu_integral_length_scale(prof.z, Z0)
fig, _ = inflow_report.plot_integral_length_scale(prof.z, L_num, REFERENCE_HEIGHT, L_theory=L_eu)
dbg.savefig(fig, f"{prof.name}/integral_length_scale.png")
plot_config.close(fig)
print(f"code-comparison figures written (cat_eu={CAT_EU}, z0={Z0})")

In [ ]:
# --- directional design speed (NBR 6123 / EN 1991-1-4) ------------------
# Design reference speed U_H per wind direction from the wind-analysis CSVs
# (real projects ship these under case_data; here an in-repo generic fixture).
from cfdmod.analytical import WindProfile_EU, WindProfile_NBR

V0 = float(os.environ.get("CFDMOD_HR_V0", "35.0"))
DESIGN_HEIGHT = float(os.environ.get("CFDMOD_HR_DESIGN_HEIGHT", "100.0"))
WIND_DIR = FIX / "inflow" / "wind_analysis"
WIND_NBR = pathlib.Path(os.environ.get("CFDMOD_HR_WIND_NBR", WIND_DIR / "wind_analysis_NBR.csv"))
WIND_EU = pathlib.Path(os.environ.get("CFDMOD_HR_WIND_EU", WIND_DIR / "wind_analysis_EU.csv"))

u_h_nbr = inflow_report.directional_reference_speed(
    WindProfile_NBR.build(WIND_NBR, V0=V0), height=DESIGN_HEIGHT, recurrence_period=50, use_kd=True
)
u_h_eu = inflow_report.directional_reference_speed(
    WindProfile_EU.build(WIND_EU, Vb=V0), height=DESIGN_HEIGHT, recurrence_period=50, use_kd=True
)
print(f"governing U_H @ {DESIGN_HEIGHT:g} m: NBR {u_h_nbr.max():.2f} @ {u_h_nbr.idxmax():g} deg | EU {u_h_eu.max():.2f} @ {u_h_eu.idxmax():g} deg")

fig, ax = plot_config.new_axes(
    xlabel="wind direction [deg]", ylabel="U_H [m/s]", title="Directional design speed"
)
ax.plot(u_h_nbr.index, u_h_nbr.to_numpy(), "-o", ms=3, label="NBR 6123")
ax.plot(u_h_eu.index, u_h_eu.to_numpy(), "-s", ms=3, label="EN 1991-1-4")
ax.legend()
dbg.savefig(fig, "directional_U_H.png", deliverable=True)
plot_config.close(fig)

table = u_h_nbr.rename("U_H_NBR").to_frame().join(u_h_eu.rename("U_H_EU"))
dbg.save_csv(table.rename_axis("wind_direction").reset_index(), "directional_U_H.csv", deliverable=True)

In [ ]:
import json

# --- persist u_ref for the Cp step (the 'update config' step) -----------
# The richest profile is the representative inlet column.
primary = profiles[0].name if profiles else None
u_ref = u_ref_by_profile.get(primary)
out = dbg.deliverable_path("reference_velocity.json")
out.write_text(
    json.dumps(
        {"profile": primary, "reference_height": REFERENCE_HEIGHT, "u_ref": u_ref},
        indent=2,
    )
)
print(f"u_ref = {u_ref} m/s -> {out}")
# In notebook 02: case = case.with_reference_velocity(u_ref)